In [2]:
# ============================================================
# NB_CNFGS_SetupLhConfig — CONFIGURACIÓN MAESTRA
# Fuente única de verdad para todos los pipelines y notebooks
# 
# ============================================================
from pyspark.sql.functions import col

# ============================================================
# 1. LIMPIEZA DE TABLAS PREVIAS
# ============================================================
print("🧹 1. LIMPIEZA DE TABLAS PREVIAS")
spark.sql("TRUNCATE TABLE lh_config.dbo.trigger_origins")
spark.sql("TRUNCATE TABLE lh_config.dbo.origin_bronze_silver")
spark.sql("TRUNCATE TABLE lh_config.dbo.origin_gold")
print("✅ Tablas vaciadas.")

# ============================================================
# 2. CREACIÓN DE ESTRUCTURA (si no existen)
# ============================================================
print("\n🏗️ 2. VERIFICANDO ESTRUCTURA DE TABLAS")

spark.sql("""
    CREATE TABLE IF NOT EXISTS lh_config.dbo.trigger_origins (
        pipeline_name   STRING,
        activo          INT,
        pipeline_id     STRING,
        diario_manana   INT,
        diario_tarde    INT
    ) USING DELTA
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS lh_config.dbo.origin_bronze_silver (
        activo                    INT,
        origen                    STRING,
        tipo_origen               STRING,
        nombre_tabla              STRING,
        ubicacion_relativa        STRING,
        parametros_incrementales  STRING,
        tipo_carga                STRING,
        ultimo_valor_incremental  STRING,
        notebook_bronze           STRING,
        notebook_silver           STRING,
        notebook_bronze_id        STRING,
        notebook_silver_id        STRING,
        pks                       STRING,
        separador_csv             STRING
    ) USING DELTA
""")

# FIX: DROP TABLE para forzar recreación con schema nuevo (+ modelo + nombre_tabla)
# La tabla puede existir con schema viejo (5 cols) → schema mismatch en INSERT
spark.sql("DROP TABLE IF EXISTS lh_config.dbo.origin_gold")
print("   🗑️  origin_gold eliminada (recrear con schema extendido)")

spark.sql("""
    CREATE TABLE IF NOT EXISTS lh_config.dbo.origin_gold (
        orden           INT,
        activo          INT,
        nombre_notebook STRING,
        notebook_id     STRING,
        descripcion     STRING,
        modelo          STRING,
        nombre_tabla    STRING
    ) USING DELTA
""")
print("✅ Estructuras verificadas.")

# ============================================================
# 3. INSERCIÓN DE DATOS MAESTROS
# ============================================================
print("\n💾 3. CARGANDO DATOS DE CONFIGURACIÓN...")

# ── A. trigger_origins ───────────────────────────────────────
# CORRECCIÓN: solo el orquestador maestro va aquí.
# DP_INGST_SilverSFTP y DP_INGST_SilverADVWDB son invocados POR
# DP_ORCHS_Origenes — no son triggers independientes.
spark.sql("""
    INSERT INTO lh_config.dbo.trigger_origins VALUES
    ('DP_ORCHS_Origenes', 1, '540b369c-0e96-4a01-aaf4-c8f2a9d095da', 1, 0)
""")
print("   ✅ trigger_origins cargado (1 pipeline maestro).")

# ── B. origin_bronze_silver ──────────────────────────────────
# CORRECCIÓN DE IDs:
#   SalesOrderHeader bronze: 5f7d35df-0a69-4358-a725-63d87a0f1965
#   SalesOrderHeader silver: 945c4ea0-d059-4496-9d44-cf8fe3d8767b
#   Person           bronze: 4c98416d-8e3c-4e05-a3fa-427b296c89a7
#   Person           silver: 941abb3a-e093-4a5d-9230-608d399c39ad
#   SalesOrderDetail silver: e35d0414-3967-4599-a125-88acc440b7e4
#   Product          silver: 37c7b35f-09cc-4823-9320-90d324b80e9f
#
# NOTA: notebook_bronze_id ADVWDB = NULL (ADVWDB va directo a Silver)
#       notebook_bronze = NULL (no existe capa Bronze para ADVWDB )
#       FIX APLICADO: campos notebook_bronze y notebook_bronze_id = NULL para ADVWDB
spark.sql("""
    INSERT INTO lh_config.dbo.origin_bronze_silver VALUES

    -- 1. SFTP / SalesOrderHeader — Incremental sin upsert
    --    Landing: TP_AdventureWorks/Sales/SalesOrderHeader_AAAAMMDD.csv
    --    Watermark inicial: '20260221' → procesa 20260222, 23, 24, 25
    (1, 'SFTP', 'sftp_csv', 'SalesOrderHeader',
     'TP_AdventureWorks/Sales',
     'fecha_archivo', 'incremental', '20260221',
     'NB_INGST_BronzeSFTP_SalesOrderHeader', 'NB_TRNSF_SilverSFTP_SalesOrderHeader',
     '5f7d35df-0a69-4358-a725-63d87a0f1965', '945c4ea0-d059-4496-9d44-cf8fe3d8767b',
     'SalesOrderID', ';'),

    -- 2. SFTP / Person — Carga total
    --    Landing: TP_AdventureWorks/Person/Person_AAAAMMDD.csv
    (1, 'SFTP', 'sftp_csv', 'Person',
     'TP_AdventureWorks/Person',
     NULL, 'total', NULL,
     'NB_INGST_BronzeSFTP_Person', 'NB_TRNSF_SilverSFTP_Person',
     '4c98416d-8e3c-4e05-a3fa-427b296c89a7', '941abb3a-e093-4a5d-9230-608d399c39ad',
     'BusinessEntityID', ';'),

    -- 3. ADVWDB / SalesOrderDetail — Incremental + UPSERT (MERGE)
    --    ADVWDB va directo a Silver (RTO del TP) → notebook_bronze = NULL
    -- FIX: notebook_bronze=NULL (ADVWDB va directo a Silver — RTO del TP)
    -- No existe NB_INGST_BronzeADVWDB_* porque ADVWDB omite capa Bronze.
    (1, 'ADVWDB', 'sql_db', 'SalesOrderDetail',
     'Sales',
     'ModifiedDate', 'incremental', '2014-01-01 00:00:00',
     NULL, 'NB_TRNSF_SilverADVWDB_SalesOrderDetail',
     NULL, 'e35d0414-3967-4599-a125-88acc440b7e4',
     'SalesOrderID,SalesOrderDetailID', NULL),

    -- 4. ADVWDB / Product — Carga total
    --    ADVWDB va directo a Silver (RTO del TP) → notebook_bronze = NULL
    -- FIX: notebook_bronze=NULL (ADVWDB va directo a Silver — RTO del TP)
    (1, 'ADVWDB', 'sql_db', 'Product',
     'Production',
     NULL, 'total', NULL,
     NULL, 'NB_TRNSF_SilverADVWDB_Product',
     NULL, '37c7b35f-09cc-4823-9320-90d324b80e9f',
     'ProductID', NULL)
""")
print("   ✅ origin_bronze_silver cargado (4 tablas).")

# ── C. origin_gold ───────────────────────────────────────────
# Orden estricto: Calendario → Cliente → Producto → Fact Ventas
# Calendario debe ser primero (fact_ventas hace JOIN sobre él)
spark.sql("""
    -- FIX: columnas modelo + nombre_tabla agregadas
    -- Los NB_TRNSF_GoldVentas_* leen estas columnas para construir
    -- tabla_destino = f'lh_gold.{modelo}.{nombre_tabla}' dinámicamente
    -- Convención TP Manual Arqui: lh_gold.<modelo>.<dim/fact_tabla>
    INSERT INTO lh_config.dbo.origin_gold VALUES
    (1, 1, 'NB_TRNSF_GoldVentas_DimCalendario', '6610b917-1596-4a61-ac00-2e4accdb4e08', 'Dim Calendario + API Feriados Argentina', 'ventas', 'd_calendario'),
    (2, 1, 'NB_TRNSF_GoldVentas_DimCliente',    '2cad884f-ab76-43af-acef-cfdb9a0ad0e4', 'Dim Cliente desde SFTP Person',           'ventas', 'd_cliente'),
    (3, 1, 'NB_TRNSF_GoldVentas_DimProducto',   '4cfb4992-7381-43d6-ae94-d89d368fe6d1', 'Dim Producto desde ADVWDB',               'ventas', 'd_producto'),
    (4, 1, 'NB_TRNSF_GoldVentas_FactVentas',    'edd43daa-a80b-41eb-96e2-b49a9ac99504', 'Fact Ventas + Aggregates',                'ventas', 'f_ventas')
""")
print("   ✅ origin_gold cargado (4 notebooks, orden 1→4).")

# ============================================================
# 4. VERIFICACIÓN FINAL
# ============================================================
print("\n🔍 4. VERIFICACIÓN FINAL")

print("\n--- trigger_origins ---")
display(spark.sql("SELECT * FROM lh_config.dbo.trigger_origins"))

print("\n--- origin_bronze_silver ---")
display(spark.sql("""
    SELECT origen, nombre_tabla, tipo_carga,
           ultimo_valor_incremental,
           notebook_bronze_id,
           notebook_silver_id,
           pks, separador_csv
    FROM lh_config.dbo.origin_bronze_silver
    ORDER BY origen, nombre_tabla
"""))

print("\n--- origin_gold ---")
display(spark.sql("SELECT * FROM lh_config.dbo.origin_gold ORDER BY orden"))

print("\n🎉 CONFIGURACIÓN COMPLETADA.")
print("   ✅ Configuración completa — sin pendientes de notebook_ids.")
print("   ℹ️  notebook_bronze=NULL para ADVWDB es correcto (ADVWDB va directo a Silver).")


StatementMeta(, f37c0d3e-4644-41b5-87b0-26d8d3fb279f, 4, Finished, Available, Finished, False)

🧹 1. LIMPIEZA DE TABLAS PREVIAS
✅ Tablas vaciadas.

🏗️ 2. VERIFICANDO ESTRUCTURA DE TABLAS
   🗑️  origin_gold eliminada (recrear con schema extendido)
✅ Estructuras verificadas.

💾 3. CARGANDO DATOS DE CONFIGURACIÓN...
   ✅ trigger_origins cargado (1 pipeline maestro).
   ✅ origin_bronze_silver cargado (4 tablas).
   ✅ origin_gold cargado (4 notebooks, orden 1→4).

🔍 4. VERIFICACIÓN FINAL

--- trigger_origins ---


SynapseWidget(Synapse.DataFrame, b967cab1-1384-40ed-9c5d-851a5f288b1d)


--- origin_bronze_silver ---


SynapseWidget(Synapse.DataFrame, 01022a4b-6964-4625-8fc1-8ffc06f0ded9)


--- origin_gold ---


SynapseWidget(Synapse.DataFrame, 0e196acb-9a4e-4bed-b2ea-1f64981671b9)


🎉 CONFIGURACIÓN COMPLETADA.
   ✅ Configuración completa — sin pendientes de notebook_ids.
   ℹ️  notebook_bronze=NULL para ADVWDB es correcto (ADVWDB va directo a Silver).
